# LiteLLM model examples

This notebook connects to the ERNI LiteLLM proxy with the virtual key in `.env`. It demonstrates:

- `gpt-5.6-luna` for text generation
- `text-embedding-3-small` for embeddings and similarity
- `gpt-image-2` for image generation

The notebook never prints the API key. Each model call consumes the virtual key's budget.

## One-time setup

From this repository, install the notebook and client dependencies, then start Jupyter:

```bash
uv add openai ipykernel
```

Your `.env` must contain `LITELLM_API_KEY` and `LITELLM_BASE_URL`. The latter should be the proxy root, for example `https://proxy.example.com`; the notebook adds `/v1` when needed.

In [ ]:
import base64
import math
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("LITELLM_API_KEY")
proxy_root = os.getenv("LITELLM_BASE_URL", "").rstrip("/")

if not api_key:
    raise RuntimeError("Set LITELLM_API_KEY in .env.")
if not proxy_root:
    raise RuntimeError("Set LITELLM_BASE_URL in .env.")

api_base = proxy_root if proxy_root.endswith("/v1") else f"{proxy_root}/v1"
client = OpenAI(api_key=api_key, base_url=api_base)

MODELS = {
    "text": "gpt-5.6-luna",
    "embedding": "text-embedding-3-small",
    "image": "gpt-image-2",
}

print(f"Connected to {api_base} (API key loaded but hidden).")

In [ ]:
# Confirm which models this virtual key can currently see.
available_models = sorted(model.id for model in client.models.list().data)
print(*available_models, sep="\n")

## Text generation — `gpt-5.6-luna`

The Responses API is the default API for a single text task. `output_text` provides the assembled text output.

In [ ]:
response = client.responses.create(
    model=MODELS["text"],
    instructions="You are a concise assistant for an AI engineering training course.",
    input="Explain cosine similarity in two short sentences.",
)

print(response.output_text)

## Embeddings — `text-embedding-3-small`

Embeddings turn text into vectors. This example compares two short sentences with cosine similarity.

In [ ]:
texts = [
    "A hen is drinking water.",
    "A chicken drinks from a water dispenser.",
]

embedding_response = client.embeddings.create(
    model=MODELS["embedding"],
    input=texts,
)
vectors = [item.embedding for item in embedding_response.data]

def cosine_similarity(left, right):
    numerator = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(a * a for a in left))
    right_norm = math.sqrt(sum(b * b for b in right))
    return numerator / (left_norm * right_norm)

print(f"Vector dimensions: {len(vectors[0])}")
print(f"Cosine similarity: {cosine_similarity(*vectors):.3f}")

## Image generation — `gpt-image-2`

GPT image models return Base64 image data by default. This cell displays it and saves `generated_poultry_image.png` locally.

In [ ]:
from IPython.display import Image, display

image_response = client.images.generate(
    model=MODELS["image"],
    prompt=(
        "A clean, friendly flat illustration of a chicken coop sensor dashboard, "
        "blue and green palette, no text"
    ),
    size="1024x1024",
    quality="low",
)

image_bytes = base64.b64decode(image_response.data[0].b64_json)
output_path = Path("generated_poultry_image.png")
output_path.write_bytes(image_bytes)
display(Image(data=image_bytes))
print(f"Saved {output_path.resolve()}")

## Keep the key safe

- Keep `.env` out of source control and never paste the virtual key into a notebook cell.
- The virtual key has the budget and model restrictions set in LiteLLM; a generated image or response still consumes that budget.
- Delete or rotate the key in LiteLLM if it is exposed.